# 10 - ML Filter (XGBoost) on SMC Signals
Train XGBoost to predict WIN/LOSS from SMC backtest trades, then filter signals

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

import os
from dotenv import load_dotenv
load_dotenv()
import MetaTrader5 as mt5

mt5.initialize()
login = int(os.getenv('MT5_LOGIN', '0'))
password = os.getenv('MT5_PASSWORD')
server = os.getenv('MT5_SERVER', '').strip()
mt5.login(login, password=password, server=server)

h4 = pd.DataFrame(mt5.copy_rates_from_pos('EURUSD', mt5.TIMEFRAME_H4, 0, 2500))
m15 = pd.DataFrame(mt5.copy_rates_from_pos('EURUSD', mt5.TIMEFRAME_M15, 0, 10000))
mt5.shutdown()

for df in [h4, m15]:
    df['time'] = pd.to_datetime(df['time'], unit='s')
    df.set_index('time', inplace=True)

print(f'H4:  {len(h4)} bars')
print(f'M15: {len(m15)} bars')

H4:  2500 bars
M15: 10000 bars


In [2]:
def find_swings(df):
    highs, lows = [], []
    for i in range(2, len(df)-2):
        if df['high'].iloc[i] == df['high'].iloc[i-2:i+3].max():
            highs.append({'t': df.index[i], 'p': df['high'].iloc[i]})
        if df['low'].iloc[i] == df['low'].iloc[i-2:i+3].min():
            lows.append({'t': df.index[i], 'p': df['low'].iloc[i]})
    return pd.DataFrame(highs) if highs else pd.DataFrame(), pd.DataFrame(lows) if lows else pd.DataFrame()

def get_h4_bias(df):
    if len(df) < 10:
        return 'neutral', 0
    highs, lows = find_swings(df)
    if len(highs) < 2 or len(lows) < 2:
        return 'neutral', 0
    close = df['close'].iloc[-1]
    prev_high = highs['p'].iloc[-2]
    prev_low = lows['p'].iloc[-2]
    if close > prev_high:
        return 'bullish', abs(close - prev_high) * 10000
    if close < prev_low:
        return 'bearish', abs(close - prev_low) * 10000
    return 'neutral', 0

def find_zones(df, min_gap=0.00005, impulse_min=0.0010):
    fvgs, obs = [], []
    for i in range(2, len(df)):
        c1, c2, c3 = df.iloc[i-2], df.iloc[i-1], df.iloc[i]
        if c3['low'] > c1['high']:
            fvgs.append({'t': df.index[i], 'd': 'bullish', 'top': c3['low'], 'bot': c1['high'], 'mid': (c3['low']+c1['high'])/2})
        if c3['high'] < c1['low']:
            fvgs.append({'t': df.index[i], 'd': 'bearish', 'top': c1['low'], 'bot': c3['high'], 'mid': (c1['low']+c3['high'])/2})
    for i in range(1, len(df)):
        prev, curr = df.iloc[i-1], df.iloc[i]
        if prev['close'] < prev['open'] and curr['close']-curr['open'] >= impulse_min:
            obs.append({'t': df.index[i], 'd': 'bullish', 'top': max(prev['open'],prev['close']), 'bot': min(prev['open'],prev['close']), 'mid': (max(prev['open'],prev['close'])+min(prev['open'],prev['close']))/2})
        if prev['close'] > prev['open'] and curr['open']-curr['close'] >= impulse_min:
            obs.append({'t': df.index[i], 'd': 'bearish', 'top': max(prev['open'],prev['close']), 'bot': min(prev['open'],prev['close']), 'mid': (max(prev['open'],prev['close'])+min(prev['open'],prev['close']))/2})
    return pd.DataFrame(obs) if obs else pd.DataFrame(), pd.DataFrame(fvgs) if fvgs else pd.DataFrame()

def get_confluence(obs, fvg, bias, max_dist=0.0005):
    if len(obs)==0 or len(fvg)==0:
        return pd.DataFrame()
    if bias != 'neutral':
        obs = obs[obs['d']==bias].copy(); fvg = fvg[fvg['d']==bias].copy()
    zones = []
    for _, o in obs.iterrows():
        for _, f in fvg.iterrows():
            if o['d']!=f['d']: continue
            if abs(o['mid']-f['mid'])<=max_dist:
                zones.append({'t':max(o['t'],f['t']),'d':o['d'],'top':max(o['top'],f['top']),'bot':min(o['bot'],f['bot']),'mid':(max(o['top'],f['top'])+min(o['bot'],f['bot']))/2})
    return pd.DataFrame(zones).sort_values('t') if zones else pd.DataFrame()

def get_session(t):
    h = t.hour
    if 0 <= h < 8: return 0  # Asian
    if 8 <= h < 16: return 1  # London
    return 2  # NY

def atr(df, period=14):
    if len(df) < period+1:
        return 0
    tr = pd.DataFrame({'hl': df['high']-df['low'], 'hc': abs(df['high']-df['close'].shift()), 'lc': abs(df['low']-df['close'].shift())}).max(axis=1)
    return tr.tail(period).mean()

print('Functions loaded')

Functions loaded


## 1. Backtest with Feature Collection
Same walk-forward as notebook 09, but saves feature vectors for ML training

In [3]:
SPREAD = 0.0001; BALANCE = 4654.85; RISK = 1.0
trades = []; open_trade = None; balance = BALANCE
check_every = 4; warmup_h4 = 200; warmup_m15 = 500

for i in range(warmup_m15, len(m15), check_every):
    ct = m15.index[i]
    bar = m15.iloc[i]

    if open_trade is not None:
        if open_trade['dir'] == 'buy':
            if bar['low'] <= open_trade['sl']:
                pnl = (open_trade['sl']-open_trade['entry']-SPREAD)*open_trade['lot']*100000
                open_trade.update({'exit':open_trade['sl'],'exit_time':ct,'pnl':pnl,'result':'loss'}); trades.append(open_trade); balance+=pnl; open_trade=None
            elif bar['high'] >= open_trade['tp']:
                pnl = (open_trade['tp']-open_trade['entry']-SPREAD)*open_trade['lot']*100000
                open_trade.update({'exit':open_trade['tp'],'exit_time':ct,'pnl':pnl,'result':'win'}); trades.append(open_trade); balance+=pnl; open_trade=None
        else:
            if bar['high'] >= open_trade['sl']:
                pnl = (open_trade['entry']-open_trade['sl']-SPREAD)*open_trade['lot']*100000
                open_trade.update({'exit':open_trade['sl'],'exit_time':ct,'pnl':pnl,'result':'loss'}); trades.append(open_trade); balance+=pnl; open_trade=None
            elif bar['low'] <= open_trade['tp']:
                pnl = (open_trade['entry']-open_trade['tp']-SPREAD)*open_trade['lot']*100000
                open_trade.update({'exit':open_trade['tp'],'exit_time':ct,'pnl':pnl,'result':'win'}); trades.append(open_trade); balance+=pnl; open_trade=None
    if open_trade is not None:
        continue

    h4_avail = h4[h4.index <= ct]
    if len(h4_avail) < warmup_h4: continue
    bias, bias_str = get_h4_bias(h4_avail)
    if bias == 'neutral': continue

    m15_avail = m15.iloc[:i+1]
    obs, fvgs = find_zones(m15_avail)
    zones = get_confluence(obs, fvgs, bias)
    if len(zones)==0: continue

    price = m15_avail['close'].iloc[-1]
    recent = zones[zones['t'] >= m15_avail.index[-20]]
    active = None
    for _, z in recent.iterrows():
        if z['bot']<=price<=z['top']: active=z; break
    if active is None: continue

    last10 = m15_avail.tail(10)
    signal = False
    if bias == 'bullish' and last10['low'].iloc[-1] > last10['low'].iloc[-3] and last10['high'].iloc[-1] > last10['high'].iloc[-3]:
        signal = True; direction = 'buy'
    elif bias == 'bearish' and last10['high'].iloc[-1] < last10['high'].iloc[-3] and last10['low'].iloc[-1] < last10['low'].iloc[-3]:
        signal = True; direction = 'sell'
    if not signal: continue

    # Features
    has_ob = len(obs) > 0 and bias in obs['d'].values
    has_fvg = len(fvgs) > 0 and bias in fvgs['d'].values
    zone_type = 2 if (has_ob and has_fvg) else 1 if has_fvg else 0
    session = get_session(ct)
    dow = ct.dayofweek
    sl_pips = 0
    if direction == 'buy':
        sl_pips = max(round((price - last10['low'].tail(5).min()) * 10000) + 1, 10)
        sl = price - sl_pips * 0.0001; tp = price + sl_pips * 3 * 0.0001
    else:
        sl_pips = max(round((last10['high'].tail(5).max() - price) * 10000) + 1, 10)
        sl = price + sl_pips * 0.0001; tp = price - sl_pips * 3 * 0.0001
    vol = atr(m15_avail.tail(50), 14) * 10000

    risk_amt = balance * RISK / 100
    lot = max(round(risk_amt/(sl_pips*10),2), 0.01)

    open_trade = {'entry':price,'sl':sl,'tp':tp,'dir':direction,'lot':lot,'entry_time':ct,
                  'sl_pips':sl_pips,'bias':bias,'bias_strength':bias_str,
                  'zone_dist':abs(price-active['mid'])*10000,'zone_type':zone_type,
                  'session':session,'dow':dow,'atr':round(vol,1)}

df_trades = pd.DataFrame(trades)
print(f'Trades: {len(df_trades)} ({df_trades["result"].value_counts().to_dict()})')
print(f'Net P&L: ${df_trades["pnl"].sum():.2f}')

Trades: 66 ({'loss': 45, 'win': 21})
Net P&L: $110.60


## 2. Prepare ML Dataset

In [4]:
features = ['bias_strength', 'zone_dist', 'zone_type', 'session', 'dow', 'sl_pips', 'atr']
X = df_trades[features].copy()
y = (df_trades['result'] == 'win').astype(int)

print(f'Samples: {len(X)}')
print(f'Wins: {y.sum()} / Losses: {(1-y).sum()}')
print(f'\nFeature preview:')
X.head(10)

Samples: 66
Wins: 21 / Losses: 45

Feature preview:


,bias_strength,zone_dist,zone_type,session,dow,sl_pips,atr
0,8.7,1.35,2,0,0,11,11.8
1,53.8,1.15,2,2,1,10,13.1
2,64.1,0.85,2,2,1,12,5.4
3,121.9,2.45,2,0,0,10,9.1
4,104.3,0.40,2,1,0,11,8.4
5,79.8,0.40,2,1,2,16,13.1
6,50.5,1.40,2,1,2,21,11.1
7,74.2,1.70,2,0,3,21,7.7
8,33.6,3.40,2,2,4,10,14.3
9,74.1,2.50,2,1,0,17,8.8


In [5]:
from sklearn.model_selection import cross_val_score, train_test_split
from xgboost import XGBClassifier

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

model = XGBClassifier(n_estimators=100, max_depth=2, learning_rate=0.1,
                      min_child_weight=5, subsample=0.7, colsample_bytree=0.7,
                      random_state=42, eval_metric='logloss')
model.fit(X_train, y_train)

train_acc = model.score(X_train, y_train)
test_acc = model.score(X_test, y_test)
print(f'Train accuracy: {train_acc:.3f}')
print(f'Test accuracy:  {test_acc:.3f}')

cv_scores = cross_val_score(model, X, y, cv=5)
print(f'CV accuracy:    {cv_scores.mean():.3f} (+/- {cv_scores.std():.3f})')

Train accuracy: 0.673
Test accuracy:  0.714
CV accuracy:    0.682 (+/- 0.020)


In [6]:
imp = pd.DataFrame({'feature': features, 'importance': model.feature_importances_}).sort_values('importance', ascending=False)
print('Feature Importance:')
print(imp.to_string(index=False))

Feature Importance:
      feature  importance
bias_strength         0.0
    zone_dist         0.0
    zone_type         0.0
      session         0.0
          dow         0.0
      sl_pips         0.0
          atr         0.0


## 3. Apply ML Filter — Re-run Backtest
Only take signals with confidence >= 0.6

In [7]:
THRESHOLD = 0.6
trades_f = []; open_trade = None; balance_f = BALANCE

for i in range(warmup_m15, len(m15), check_every):
    ct = m15.index[i]
    bar = m15.iloc[i]

    if open_trade is not None:
        if open_trade['dir'] == 'buy':
            if bar['low'] <= open_trade['sl']:
                pnl = (open_trade['sl']-open_trade['entry']-SPREAD)*open_trade['lot']*100000
                open_trade.update({'exit':open_trade['sl'],'exit_time':ct,'pnl':pnl,'result':'loss'}); trades_f.append(open_trade); balance_f+=pnl; open_trade=None
            elif bar['high'] >= open_trade['tp']:
                pnl = (open_trade['tp']-open_trade['entry']-SPREAD)*open_trade['lot']*100000
                open_trade.update({'exit':open_trade['tp'],'exit_time':ct,'pnl':pnl,'result':'win'}); trades_f.append(open_trade); balance_f+=pnl; open_trade=None
        else:
            if bar['high'] >= open_trade['sl']:
                pnl = (open_trade['entry']-open_trade['sl']-SPREAD)*open_trade['lot']*100000
                open_trade.update({'exit':open_trade['sl'],'exit_time':ct,'pnl':pnl,'result':'loss'}); trades_f.append(open_trade); balance_f+=pnl; open_trade=None
            elif bar['low'] <= open_trade['tp']:
                pnl = (open_trade['entry']-open_trade['tp']-SPREAD)*open_trade['lot']*100000
                open_trade.update({'exit':open_trade['tp'],'exit_time':ct,'pnl':pnl,'result':'win'}); trades_f.append(open_trade); balance_f+=pnl; open_trade=None
    if open_trade is not None:
        continue

    h4_avail = h4[h4.index <= ct]
    if len(h4_avail) < warmup_h4: continue
    bias, bias_str = get_h4_bias(h4_avail)
    if bias == 'neutral': continue

    m15_avail = m15.iloc[:i+1]
    obs, fvgs = find_zones(m15_avail)
    zones = get_confluence(obs, fvgs, bias)
    if len(zones)==0: continue

    price = m15_avail['close'].iloc[-1]
    recent = zones[zones['t'] >= m15_avail.index[-20]]
    active = None
    for _, z in recent.iterrows():
        if z['bot']<=price<=z['top']: active=z; break
    if active is None: continue

    last10 = m15_avail.tail(10)
    signal = False; direction = None
    if bias == 'bullish' and last10['low'].iloc[-1] > last10['low'].iloc[-3] and last10['high'].iloc[-1] > last10['high'].iloc[-3]:
        signal=True; direction='buy'
    elif bias == 'bearish' and last10['high'].iloc[-1] < last10['high'].iloc[-3] and last10['low'].iloc[-1] < last10['low'].iloc[-3]:
        signal=True; direction='sell'
    if not signal: continue

    has_ob = len(obs) > 0 and bias in obs['d'].values
    has_fvg = len(fvgs) > 0 and bias in fvgs['d'].values
    zone_type = 2 if (has_ob and has_fvg) else 1 if has_fvg else 0
    session = get_session(ct); dow = ct.dayofweek
    sl_pips = round(((price-last10['low'].tail(5).min()) if direction=='buy' else (last10['high'].tail(5).max()-price))*10000)+1
    sl_pips = max(sl_pips, 10)
    vol = atr(m15_avail.tail(50), 14)*10000

    feat = pd.DataFrame([[bias_str, abs(price-active['mid'])*10000, zone_type, session, dow, sl_pips, round(vol,1)]], columns=features)
    conf = model.predict_proba(feat)[0][1]
    if conf < THRESHOLD:
        continue

    risk_amt = balance_f * RISK / 100
    lot = max(round(risk_amt/(sl_pips*10),2), 0.01)
    sl = (price - sl_pips*0.0001) if direction=='buy' else (price + sl_pips*0.0001)
    tp = (price + sl_pips*3*0.0001) if direction=='buy' else (price - sl_pips*3*0.0001)
    open_trade = {'entry':price,'sl':sl,'tp':tp,'dir':direction,'lot':lot,'entry_time':ct,'sl_pips':sl_pips,'confidence':round(conf,3)}

df_f = pd.DataFrame(trades_f)
print('--- Unfiltered vs Filtered ---')
print(f'Unfiltered: {len(df_trades)} trades, ${df_trades["pnl"].sum():.2f}')
filtered_pnl = df_f['pnl'].sum() if len(df_f) > 0 else 0
print(f'Filtered:   {len(df_f)} trades, ${filtered_pnl:.2f}')

--- Unfiltered vs Filtered ---
Unfiltered: 66 trades, $110.60
Filtered:   0 trades, $0.00


## 4. Compare Results

In [8]:
def report(df, label):
    if len(df)==0: print(f'{label}: No trades'); return
    w=len(df[df['result']=='win']); l=len(df[df['result']=='loss']); n=df['pnl'].sum()
    gp=df[df['pnl']>0]['pnl'].sum(); gl=abs(df[df['pnl']<0]['pnl'].sum())
    pf=gp/gl if gl>0 else float('inf')
    wr=w/len(df)*100
    print(f'--- {label} ---')
    print(f'Trades: {len(df)} ({w}W/{l}L) | WR: {wr:.1f}% | PF: {pf:.2f} | P&L: ${n:.2f}')

report(df_trades, 'Unfiltered')
report(df_f, f'Filtered (conf>={THRESHOLD})')

--- Unfiltered ---
Trades: 66 (21W/45L) | WR: 31.8% | PF: 1.05 | P&L: $110.60
Filtered (conf>=0.6): No trades


In [9]:
print('\n--- Confidence Distribution (Filtered Trades) ---')
if len(df_f) > 0:
    for _, t in df_f.iterrows():
        print(f"  {t['entry_time']} | {t['dir']} | conf={t['confidence']:.3f} | {t['result']} | ${t['pnl']:.2f}")
else:
    print('No trades passed the filter')


--- Confidence Distribution (Filtered Trades) ---
No trades passed the filter
